Extração de eventos futuros

In [0]:
# ----------------------------------------------------------------------------------------
# SCRIPT: 2.5_eventos_futuros_ingest
# LOCAL: 1_bronze/src/2_calendario_eventos
# OBJETIVO: Foco em eventos com datas posteriores a hoje 
# ENTREGA: View pública com calendário de eventos futuros já agendados
# ----------------------------------------------------------------------------------------

import requests as req
from datetime import date

# Configurações Iniciais
data_hoje = date.today().strftime('%Y-%m-%d')
data_fim = "2026-07-31"
pagina = 1
todos_eventos = []

print(f"Iniciando captura de eventos futuros a partir de {data_hoje}...")

# Loop de Paginação
while True:
    url = f"https://dadosabertos.camara.leg.br/api/v2/eventos?dataInicio={data_hoje}&dataFim={data_fim}&pagina={pagina}&itens=100&ordem=ASC&ordenarPor=dataHoraInicio"
    
    try:
        res = req.get(url).json()
        dados_da_pagina = res['dados']

        # Se página vazia, interrompe o loop
        if not dados_da_pagina:
            break
            
        todos_eventos.extend(dados_da_pagina)
        print(f"Página {pagina} capturada com {len(dados_da_pagina)} eventos...")
        
        pagina += 1
        
    except Exception as e:
        print(f"Erro na captura da página {pagina}: {e}")
        break

# Salvar na Bronze
if todos_eventos:
    
    esquema_referencia = spark.table("bronze_eventos").schema
    
    df_futuros = spark.createDataFrame(todos_eventos, schema=esquema_referencia)
    
    df_futuros.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze_eventos_futuros")
    
    print(f"Sucesso! Total de {len(todos_eventos)} eventos salvos.")
else:
    print("Nenhum evento futuro encontrado.")

display(spark.table("bronze_eventos_futuros"))